# Libraries and global variables

In [ ]:
import sys
import time
from pathlib import Path
import json

import numpy as np

import torch
from fastai.vision.all import (
    DataBlock, ImageBlock,
    ColReader, ColSplitter,
    vision_learner,
    SaveModelCallback, EarlyStoppingCallback, CSVLogger,
    accuracy, RocAuc, F1Score, ClassificationInterpretation,
    Resize, DataLoaders
)
from fastai.callback.fp16 import MixedPrecision
import timm  # needed for EfficientNet via fastai

from fastai.vision.augment import (
    Brightness, Contrast, aug_transforms
)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
!cp -r "/content/drive/MyDrive/Work/10 Foodbegood/project_gn_food_estimator_v1" "/content/project_gn_food_estimator_v1"

## Directory layout & paths

In [ ]:
PLATFORM = "colab" # "colab", "kaggle", "local"

if PLATFORM == "colab":
    ROOT = Path("/content/project_gn_food_estimator_v1")
elif PLATFORM == "kaggle":
    ROOT = Path("/kaggle/working/project_gn_food_estimator")
else:
    ROOT = Path(".").resolve()

In [ ]:
DATA_DIR      = ROOT / "data"
RAW_DIR       = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
EXPORTS_DIR   = DATA_DIR / "exports"
MODELS_DIR    = ROOT / "models"
TRAINING_DIR  = ROOT / "training"
LOGS_DIR      = TRAINING_DIR / "logs"

# Labels file
COCO_MODEL2_JSON = EXPORTS_DIR / "model2_fill_level.json"

# Model checkpoint path
MODEL2_PATH = MODELS_DIR / "model2_fill.pth"

log_path = LOGS_DIR / "model2_training_log.csv"

In [ ]:
def setup_directories():
    """
    Ensure output directories exist.
    """
    MODELS_DIR.mkdir(parents = True, exist_ok = True)
    LOGS_DIR.mkdir(parents = True, exist_ok = True)
    print(f"[Setup] Models dir: {MODELS_DIR}")
    print(f"[Setup] Logs dir: {LOGS_DIR}")

In [ ]:
setup_directories()

## Parameters

In [ ]:
# Dataset split ratios
SPLIT_TRAIN = 0.80
SPLIT_VAL   = 0.10
SPLIT_TEST  = 0.10
RANDOM_SEED = 42

In [ ]:
# Class labels — must match Label Studio annotation labels exactly
MODEL2_CLASSES = ["empty", "low", "medium", "high", "full"]

In [ ]:
# If classifier confidence falls below this threshold, ask user to retake photo
CONFIDENCE_THRESHOLD = 0.70

In [ ]:
# Image pre-processing
IMAGE_SIZE = 512
NORM_MEAN = [0.485, 0.456, 0.406] # ImageNet statistics
NORM_STD = [0.229, 0.224, 0.225]
HIST_EQ_ENABLED = True

In [ ]:
# Data augmentation toggles
AUG_BRIGHTNESS = True
AUG_ROTATION   = True
AUG_BLUR       = True
AUG_SCALING    = True
AUG_FLIPPING   = True
AUG_CONTRAST   = True

## Training hyperparameters

In [ ]:
# Model #2 — EfficientNet-B0, image classification
MODEL2_BACKBONE        = "efficientnet_b0"
MODEL2_BATCH_SIZE      = 16      # Lighter model, larger batch fine on T4
MODEL2_EPOCHS_FROZEN   = 5
MODEL2_EPOCHS_UNFROZEN = 10
MODEL2_LR_FROZEN       = 1e-3
MODEL2_LR_UNFROZEN     = 1e-4
MODEL2_MIXED_PREC      = True

In [ ]:
def check_gpu():
    """
    Report GPU availability and VRAM.
    """
    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9

        print(f"[GPU] {gpu_name} - {vram_gb:.1f} GB VRAM")

        if vram_gb < 8.0:
            print(
                f"[GPU] WARNING: Less than 8 GB VRAM detected. "
                f"Consider reducing MODEL2_BATCH_SIZE."
                f" (currently {MODEL2_BATCH_SIZE})."
            )
    else:
        print("[GPU] No CUDA device found - training will run on CPU.")
    return "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
DEVICE = check_gpu()

# Data Preprocessing

## Data augmentation pipeline

In [ ]:
def get_classification_transforms(size: int = IMAGE_SIZE):
    """
    fastai's aug_transforms() covers: flip, rotation, zoom (scaling),
    warp, lighting (brightness + contrast), and blur.

    Returns
        tuple[list, list]
            (train_transforms, valid_transforms)
    """
    aug_kwargs = dict(
        size = size,

        # Flipping: only horizontal — vertical flip is disabled for top-down food
        do_flip = AUG_FLIPPING,
        flip_vert = False,

        # Rotation: ±20° captures realistic hand-angle variation
        max_rotate = 20.0 if AUG_ROTATION else 0.0,

        # Scaling / zoom: 1.0–1.15 — mild zoom to simulate distance variation
        min_zoom = 1.0,
        max_zoom = 1.15 if AUG_SCALING else 1.0,

        # Lighting (brightness + contrast handled together by aug_transforms)
        max_lighting = 0.3 if (AUG_BRIGHTNESS or AUG_CONTRAST) else 0.0,

        # Blur: p = 0.3 means applied to ~30% of batches
        max_blur = 2.0 if AUG_BLUR else 0.0,

        # Warp: small perspective warp simulates slight camera tilt
        max_warp = 0.1,

        p_affine = 0.75,   # probability of applying affine transforms
        p_lighting = 0.75  # probability of applying lighting transforms
    )

    train_tfms = aug_transforms(**aug_kwargs)
    valid_tfms = []  # No augmentation on validation / test sets

    return train_tfms, valid_tfms

In [ ]:
train_tfms, _ = get_classification_transforms(size = IMAGE_SIZE)

# Loading data

## Build DataLoaders

In [ ]:
def _load_coco_json(json_path: Path) -> dict:
    """
    Load and validate a COCO JSON file exported from Label Studio.
    """
    if not json_path.exists():
        raise FileNotFoundError(
            f"COCO annotation file not found: {json_path}\n"
            f"Export your Label Studio project as COCO JSON and place it at "
            f"the path defined in {COCO_MODEL1_JSON}."
        )
    with open(json_path, "r") as f:
        data = json.load(f)
    required_keys = {"images", "annotations", "categories"}
    if not required_keys.issubset(data.keys()):
        raise ValueError(
            f"COCO JSON at {json_path} is missing keys: "
            f"{required_keys - data.keys()}"
        )

    return data


def _split_image_ids(
        image_ids: list[int],
        train_ratio: float = SPLIT_TRAIN,
        val_ratio: float = SPLIT_VAL,
        seed: int = RANDOM_SEED,
    ) -> tuple[set[int], set[int], set[int]]:
    """
    Randomly split a list of image IDs into train / val / test sets.

    Returns:
        tuple[set, set, set]
            (train_ids, val_ids, test_ids)
    """
    rng = np.random.default_rng(seed)
    ids = np.array(image_ids)
    rng.shuffle(ids)

    n = len(ids)
    n_train = int(n * train_ratio)
    n_val = int(n * val_ratio)

    train_ids = set(ids[:n_train].tolist())
    val_ids = set(ids[n_train:n_train + n_val].tolist())
    test_ids = set(ids[n_train + n_val:].tolist())

    print(
        f"Split -> train: {len(train_ids)} | val: {len(val_ids)} | test: {len(test_ids)} images"
    )

    return train_ids, val_ids, test_ids

In [ ]:
def get_classification_dataloaders(
    coco_json_path: Path  = COCO_MODEL2_JSON,
    image_dir:      Path  = PROCESSED_DIR,  # cropped container images
    image_size:     int   = IMAGE_SIZE,
    batch_size:     int   = MODEL2_BATCH_SIZE,
    train_tfms:     list  = None,
    device:         str   = "cuda",
    verbose:        bool  = True
    ) -> tuple[DataLoaders, pd.DataFrame, pd.DataFrame]:
    """
    Build fastai DataLoaders for fill-level image classification (Model #2).

    Note: image_dir defaults to PROCESSED_DIR because the classification
    images should be cropped to show only the container interior
    (as specified in your Label Studio setup).

    Parameters
    ----------
    coco_json_path : Path
        Path to the COCO JSON classification annotation file.
    image_dir : Path
        Directory containing cropped container images.
    image_size : int
        Target image size.
    batch_size : int
        Training batch size.
    train_tfms : list, optional
        Fastai transforms for training set.
    device : str
        "cuda" or "cpu".
    verbose : bool
        Print split and class information.

    Returns
    -------
    tuple[DataLoaders, pd.DataFrame, pd.DataFrame]
        (dls, train_df, test_df)
    """
    if verbose:
        print(f"Loading COCO annotations from: {coco_json_path}")

    coco = _load_coco_json(coco_json_path)

    id_to_label = {cat["id"]: cat["name"] for cat in coco["categories"]}
    if verbose:
        print(f"Categories: {id_to_label}")

        coco_labels = set(id_to_label.values())
        cfg_labels  = set(MODEL2_CLASSES)
        if coco_labels != cfg_labels:
            print(
                f"WARNING: COCO labels {coco_labels} differ from "
                f"MODEL2_CLASSES {cfg_labels}. "
                f"Update MODEL2_CLASSES if needed."
            )

    # Build image ID -> filename mapping
    id_to_file = {img["id"]: Path(img["file_name"].replace("\\", "/")).name.split("-", 1)[1] for img in coco["images"]}

    # For classification, one annotation per image is expected.
    # Build image_id → category mapping from annotations.
    id_to_category = {
        ann["image_id"]: id_to_label[ann["category_id"]]
        for ann in coco["annotations"]
    }
    all_image_ids = list(id_to_file.keys())

    if verbose:
        print(f"Total images: {len(all_image_ids)}")

    train_ids, val_ids, test_ids = _split_image_ids(all_image_ids)

    rows = []
    for img_id, filename in id_to_file.items():
        label = id_to_category.get(img_id, None)
        if label is None:
            print(f"WARNING: No annotation for image id {img_id} — skipping.")
            continue
        split = (
            "train" if img_id in train_ids else
            "valid" if img_id in val_ids else
            "test"
        )
        rows.append({
            "image_path": str(image_dir / filename),
            "label": label,
            "split": split
        })

    full_df = pd.DataFrame(rows)
    test_df  = full_df[full_df["split"] == "test"].copy()
    train_df = full_df[full_df["split"] != "test"].copy()
    train_df = train_df.copy()
    train_df["is_valid"] = train_df["split"] == "valid"

    if verbose:
        print(f"Images → train + val: {len(train_df)} | test: {len(test_df)}")
        print(f"Label distribution (train + val):\n{train_df['label'].value_counts()}")

    dblock = DataBlock(
        blocks = (ImageBlock, CategoryBlock),
        get_x = ColReader("image_path"),
        get_y = ColReader("label"),
        splitter = ColSplitter("is_valid"),
        item_tfms = Resize(image_size),
        batch_tfms = train_tfms
    )

    dls = dblock.dataloaders(
        train_df,
        bs = batch_size,
        device = device
    )

    if verbose:
        print(f"DataLoaders ready — vocab: {dls.vocab}")

    return dls, train_df, test_df

In [ ]:
# Building DataLoaders
dls, train_df, test_df = get_classification_dataloaders(
        coco_json_path = COCO_MODEL2_JSON,
        image_dir = PROCESSED_DIR,
        image_size = IMAGE_SIZE,
        batch_size = MODEL2_BATCH_SIZE,
        train_tfms = train_tfms,
        device = DEVICE,
        verbose = True
    )

In [ ]:
# Verify vocab matches config
if list(dls.vocab) != MODEL2_CLASSES:
    print(
        f"\n  WARNING: DataLoaders vocab {list(dls.vocab)} does not match "
        f"MODEL2_CLASSES {MODEL2_CLASSES}.\n"
        f"  Predictions will use DataLoaders vocab ordering."
    )

In [ ]:
# Save test split
test_csv_path = LOGS_DIR / "model2_test_split.csv"
test_df.to_csv(test_csv_path, index = False)
print(f"Test split saved → {test_csv_path}")

In [ ]:
# Sample batch sanity check
xb, yb = dls.one_batch()
print(f"Sample batch — images: {xb.shape}, labels: {yb.shape}")

In [ ]:
# Class weight helper
def compute_class_weights(train_df, label_col: str = "label") -> torch.Tensor:
    """
    Compute inverse-frequency class weights to handle any class imbalance.

    With only 100 photos split 80/10/10 across 5 classes × 2 container
    types × 2 food types, some fill levels may be slightly underrepresented.
    Passing weights to CrossEntropyLossFlat helps.

    Returns
        torch.Tensor of shape (n_classes,)
    """
    counts = train_df[train_df["split"] == "train"][label_col].value_counts()

    # Ensure ordering matches MODEL2_CLASSES
    ordered_counts = [counts.get(cls, 1) for cls in MODEL2_CLASSES]
    total = sum(ordered_counts)
    weights = torch.tensor(
        [total / (len(MODEL2_CLASSES) * c) for c in ordered_counts],
        dtype = torch.float32
    )

    print("Class weights:")
    for cls, w in zip(MODEL2_CLASSES, weights):
        print(f"    {cls:8s} : {w:.3f}")

    return weights

In [ ]:
# Computing class weights (handle minor imbalance)
class_weights = compute_class_weights(train_df)
if device == "cuda":
    class_weights = class_weights.cuda()

# Model

Train Model #2: Fill level classifier (empty / low / medium / high / full)

**Architecture:**

EfficientNet-B0 backbone (ImageNet pre-trained) via timm, loaded through fastai's vision_learner with a classification head for 5 output classes.

**Strategy**

Two-phase training
- Phase 1 — backbone frozen, head trained for MODEL2_EPOCHS_FROZEN
- Phase 2 — full network unfrozen, fine-tuned with discriminative learning rates for MODEL2_EPOCHS_UNFROZEN

Mixed precision: fp16 enabled (safe on T4)

Important note on input images:
- Model #2 expects CROPPED images showing only the container interior.
- These should live in data/processed/ (not data/raw/).
- For training, make sure your Label Studio export references the cropped images, not the full-scene photos.

In [ ]:
# Metrics
# F1Score with average="macro" is important for ordinal multi-class
# problems — it treats all classes equally regardless of frequency.
metrics = [
    accuracy,
    F1Score(average ="macro")
]

## Build learner

In [ ]:
learn = vision_learner(
    dls,
    "efficientnet_b0",          # timm model name
    metrics = metrics,
    loss_func = torch.nn.CrossEntropyLoss(weight = class_weights),
    model_dir = MODELS_DIR,
    cbs = [
        SaveModelCallback(
            monitor = "valid_loss",
            fname = "model2_best",
            with_opt = True
        ),
        EarlyStoppingCallback(
            monitor = "valid_loss",
            patience = 5
        ),
        CSVLogger(fname = str(log_path))
    ]
)

## Mixed precision scaler

In [ ]:
# Mixed precision
if MODEL2_MIXED_PREC and device == "cuda":
    learn = learn.to_fp16()
    print("Mixed precision (fp16) — ENABLED")

# Training loop

## Phase 1 - frozen backbone

In [ ]:
learn.freeze()

In [ ]:
# Running lr_find (Phase 1)
suggested_lr = MODEL2_LR_FROZEN
try:
    lr_finder = learn.lr_find(suggest_funcs = None, show_plot = False)
    suggested_lr = lr_finder.valley
    print(f"lr_find suggestion : {suggested_lr:.2e}  "
            f"(config default: {MODEL2_LR_FROZEN:.2e})")
except Exception as e:
    print(f"lr_find failed ({e}), using config default: {suggested_lr:.2e}")

In [ ]:
t0 = time.time()
learn.fit_one_cycle(
    MODEL2_EPOCHS_FROZEN,
    lr_max = suggested_lr
)

print(f"  Phase 1 complete — {time.time() - t0:.0f}s")

## Phase 2 - full fine-tune

In [ ]:
learn.unfreeze()

In [ ]:
lr_unfrozen = MODEL2_LR_UNFROZEN
print(f"LR slice : {lr_unfrozen/100:.2e} → {lr_unfrozen:.2e}")

In [ ]:
t0 = time.time()
learn.fit_one_cycle(
    MODEL2_EPOCHS_UNFROZEN,
    lr_max = slice(lr_unfrozen / 100, lr_unfrozen)
)
print(f"  Phase 2 complete — {time.time() - t0:.0f}s")

# Export model

In [ ]:
learn.export(cfg.MODEL2_PATH)
print(f"\n[Done] Model exported → {MODEL2_PATH}")

state_dict_path = MODELS_DIR / "model2_state_dict.pth"
torch.save(learn.model.state_dict(), state_dict_path)
print(f"[Done] State dict → {state_dict_path}")

# Validation metrics + confusion matrix

In [ ]:
val_results = learn.validate()
print(f"valid_loss : {val_results[0]:.4f}")

for i, m in enumerate(learn.metrics):
    print(f"  {m.name:15s}: {val_results[i + 1]:.4f}")

In [ ]:
print("Confusion matrix (validation set):")
interp = ClassificationInterpretation.from_learner(learn)
interp.print_classification_report()

## Confidence threshold check on validation set

In [ ]:
print(f"Checking confidence threshold ({CONFIDENCE_THRESHOLD}):")
preds, targets = learn.get_preds(with_decoded = False)
confidences = preds.max(dim = 1).values
below_threshold = (confidences < CONFIDENCE_THRESHOLD).sum().item()
pct = below_threshold / len(confidences) * 100

print(
    f"  {below_threshold}/{len(confidences)} validation images "
    f"({pct:.1f}%) fall below the confidence threshold of "
    f"{CONFIDENCE_THRESHOLD}.\n"
    f"  These would trigger a 'retake photo' prompt in production."
)

In [ ]:
print(f"[Info] Training log → {log_path}")
print(f"[Info] Test split   → {test_csv_path}")

# Quick inference check

## Visualisation